# Deployed model report (lstm_raw v2): confusion matrix and loss curves

This notebook changes nothing about the deployed model. It loads the eight saved v2 weight files, confirms they
reproduce the deployed results, draws the confusion matrix at the deployed threshold (0.528), and retrains the same
setting with validation tracking to show how training and validation loss behave over its 15 epochs.

**Before running**: put the deployed model files in Google Drive under `My Drive/flood_tuning/v2_model/`, that is
`config.json` and `lstm_raw_seed0.weights.h5` to `lstm_raw_seed7.weights.h5` from `models/flood_lstm/` in the
project folder. `features_daily_audited.csv` stays in `My Drive/flood_tuning/` as before.

Run with **Runtime > Run all** on a T4 GPU in a fresh session (about 15 minutes).

v2 setting: LSTM with 32 units, dropout 0.3, a 14-day window, learning rate 0.001, batch size 64, 15 epochs,
event weights, trained on 2015-2023, threshold 0.528 chosen earlier on the development folds.

In [ ]:
# 1. Setup
import os, json, time, zipfile
os.environ["TF_DETERMINISTIC_OPS"] = "1"
import numpy as np, pandas as pd, tensorflow as tf
tf.get_logger().setLevel("ERROR")
from numpy.lib.stride_tricks import sliding_window_view
print("TensorFlow", tf.__version__, "| Keras", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "")
print("GPU:", tf.config.list_physical_devices("GPU") or "none (Runtime > Change runtime type > T4 GPU)")

In [ ]:
# 2. Data: upload features_daily_audited.csv from
#    data_collection/data/08_model_ready/ in the project folder, or point DATA_PATH at a copy in Google Drive.
from google.colab import drive, files
DATA_PATH = "/content/drive/MyDrive/flood_tuning/features_daily_audited.csv"   # optional Drive copy
OUT_DIR = "/content/drive/MyDrive/flood_tuning/v2_report"                       # results saved here if Drive is used
try:
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted:", e)
if not os.path.exists(DATA_PATH):
    print("Upload features_daily_audited.csv")
    up = files.upload()
    DATA_PATH = "/content/" + next(iter(up))
    OUT_DIR = "/content/v2_report"
os.makedirs(OUT_DIR, exist_ok=True)
RAW = pd.read_csv(DATA_PATH, parse_dates=["date"]).sort_values(["catchment", "date"])
print(len(RAW), "rows,", RAW["date"].min().date(), "to", RAW["date"].max().date())
print(RAW["label_status"].value_counts().to_dict())

In [ ]:
# 3. Shared code, copied from the repository (sequences.py, save_final_model.py, event_train_evaluate.py)
SEQ_FEATS = ["rain_mm", "imerg_max_rate", "imerg_total_mm", "imerg_hours_ge5", "era5_rain_mm", "soil_1", "soil_2",
             "temp_c", "dewpoint_depression", "pressure_hpa", "month_sin", "month_cos"]
CATCHMENTS = ["ngong", "mathare", "nairobi"]
DEV_FOLDS = [(range(2015, 2019), [2019]), (range(2015, 2021), [2021, 2022]), (range(2015, 2023), [2023])]
FINAL_FOLD = (range(2015, 2024), [2024, 2025])
SEEDS = list(range(8))
DAY = pd.Timedelta(days=1)
QS = np.round(np.arange(0.90, 0.9991, 0.001), 3)
_windows = {}


def make_windows(t, drop=()):
    # one window per catchment-day: the t days ending on that day, labelled with the last day
    if (t, drop) not in _windows:
        df = RAW.copy()
        for c in CATCHMENTS:
            df[f"is_{c}"] = (df["catchment"] == c).astype("float32")
        feats = [f for f in SEQ_FEATS + [f"is_{c}" for c in CATCHMENTS] if f not in drop]
        xs, metas = [], []
        for c, g in df.groupby("catchment"):
            vals = g[feats].values.astype("float32")
            xs.append(sliding_window_view(vals, (t, vals.shape[1]))[:, 0])
            metas.append(g.iloc[t - 1:][["catchment", "date", "label_status", "event_id"]].copy())
        meta = pd.concat(metas).reset_index(drop=True)
        meta["year"] = meta["date"].dt.year
        _windows[(t, drop)] = (np.concatenate(xs), meta, feats)
    return _windows[(t, drop)]


def make_model(n_feats, t, units, dropout, lr):
    m = tf.keras.Sequential([
        tf.keras.layers.Input((t, n_feats)),
        tf.keras.layers.LSTM(units),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="binary_crossentropy")
    return m


def fold_data(X, meta, train_years, val_years):
    st = meta["label_status"].values
    flood, known = st == "flood", np.isin(st, ["flood", "no_flood"])
    tr = meta["year"].isin(list(train_years)).values & known
    va = meta["year"].isin(list(val_years)).values
    flat = X[tr].reshape(-1, X.shape[2])
    mu, sd = flat.mean(0), flat.std(0) + 1e-6
    w = float(np.sqrt((tr & ~flood).sum() / (tr & flood).sum()))          # class weight
    eid = meta["event_id"].where(flood)
    L = meta[tr & flood].groupby(eid[tr & flood]).size()                   # days per event
    sw = np.where(flood, w * L.mean() / eid.map(L).fillna(1).values, 1.0)  # event weights
    return (X[tr] - mu) / sd, flood[tr].astype("float32"), sw[tr], (X[va] - mu) / sd, va, mu, sd


def run_setting(t, units, dropout, lr, max_ep, folds, drop=(), keep_models=False, log=None):
    # eight seeds; scores after every epoch, averaged over seeds -> {epoch: DataFrame(val, catchment, date, score)}
    X, meta, feats = make_windows(t, drop)
    out, models = {}, []
    for train_years, val_years in folds:
        Xtr, ytr, sw, Xva, va, mu, sd = fold_data(X, meta, train_years, val_years)
        S = np.zeros((max_ep, len(Xva)))
        for seed in SEEDS:
            tf.keras.utils.set_random_seed(seed)
            m = make_model(X.shape[2], t, units, dropout, lr)
            for ep in range(max_ep):
                h = m.fit(Xtr, ytr, sample_weight=sw, epochs=1, batch_size=64, verbose=0)
                S[ep] += m.predict(Xva, batch_size=1024, verbose=0).ravel() / len(SEEDS)
                if log is not None:
                    log.append({"val": "-".join(map(str, val_years)), "seed": seed, "epoch": ep + 1,
                                "loss": h.history["loss"][0]})
            if keep_models:
                models.append(m)
        base = meta.loc[va, ["catchment", "date"]].reset_index(drop=True)
        for ep in range(max_ep):
            d = base.copy(); d["val"] = "-".join(map(str, val_years)); d["score"] = S[ep]
            out.setdefault(ep + 1, []).append(d)
    out = {ep: pd.concat(v, ignore_index=True) for ep, v in out.items()}
    return (out, models, feats, mu, sd) if keep_models else out


LAB = RAW[["catchment", "date", "label_status", "event_id"]]


def prepare(d):
    m = d.merge(LAB, on=["catchment", "date"])
    parts = []
    for _, g in m.groupby("val"):
        ev = (g[g["label_status"] == "flood"].groupby("event_id")
              .agg(catchment=("catchment", "first"), start=("date", "min"), end=("date", "max"), days=("date", "size")))
        known = g[g["label_status"].isin(["flood", "no_flood"])]
        parts.append((g, ev, known["label_status"].eq("flood").values, known["score"].values))
    return parts


def ap(y, s):
    o = np.argsort(-s, kind="mergesort")
    y, s = y[o], s[o]
    last = np.r_[np.diff(s) != 0, True]
    tp = np.cumsum(y)[last]
    return float(np.sum(np.diff(np.r_[0, tp / tp[-1]]) * tp / np.arange(1, len(y) + 1)[last]))


def pooled_ap(parts):
    return ap(np.concatenate([p[2] for p in parts]), np.concatenate([p[3] for p in parts]))


def stats(parts, thr, cy):
    n = det = det1 = n1 = eps = false_eps = tp = fp = fn = 0
    for g, ev, y, s in parts:
        a = s >= thr
        tp += int((a & y).sum()); fp += int((a & ~y).sum()); fn += int((~a & y).sum())
        alarms = g[g["score"] >= thr]
        for e in ev.itertuples():
            x = alarms[alarms["catchment"] == e.catchment]
            hit = bool(((x["date"] >= e.start - DAY) & (x["date"] <= e.end + DAY)).any())
            det += hit
            if e.days == 1:
                n1 += 1; det1 += hit
        n += len(ev)
        for c, x in alarms.sort_values("date").groupby("catchment"):
            evc = ev[ev["catchment"] == c]
            for _, ep in x.groupby((x["date"].diff() != DAY).cumsum()):
                eps += 1
                false_eps += not ((evc["start"] - DAY <= ep["date"].max()) & (evc["end"] + DAY >= ep["date"].min())).any()
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn)
    er = det / n
    ep_p = (eps - false_eps) / eps if eps else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0,
            "events": n, "detected": det, "one_day_detected": f"{det1}/{n1}", "multi_day_detected": f"{det - det1}/{n - n1}",
            "event_recall": er, "episodes": eps, "false_episodes": false_eps, "episode_precision": ep_p,
            "event_f1": 2 * ep_p * er / (ep_p + er) if ep_p + er else 0.0, "false_eps_per_cy": false_eps / cy}


def pick_threshold(dev_scores):
    dp = prepare(dev_scores)
    grid = np.unique(np.quantile(dev_scores["score"], QS))
    sw = pd.DataFrame([{"threshold": t, **stats(dp, t, 12)} for t in grid])
    return sw.sort_values(["f1", "threshold"], ascending=[False, False]).iloc[0]


def wilson(k, n, z=1.96):
    p = k / n; c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return round(c - h, 2), round(c + h, 2)


def loss_history(t, units, dropout, lr, max_ep, fold, seeds, drop=()):
    # training and validation loss per epoch on one fold, averaged over seeds; both weighted the same way
    X, meta, _ = make_windows(t, drop)
    Xtr, ytr, sw, Xva, va, _, _ = fold_data(X, meta, *fold)
    st = meta["label_status"].values
    flood = st == "flood"
    tr_years = meta["year"].isin(list(fold[0])).values & np.isin(st, ["flood", "no_flood"])
    w = float(np.sqrt((tr_years & ~flood).sum() / (tr_years & flood).sum()))
    eid = meta["event_id"].where(flood)
    L = meta[va & flood].groupby(eid[va & flood]).size()
    known = np.isin(st[va], ["flood", "no_flood"])
    yva = flood[va].astype("float32")
    swva = np.where(yva == 1, w * L.mean() / eid[va].map(L).fillna(1).values, 1.0)
    hist = []
    for seed in seeds:
        tf.keras.utils.set_random_seed(seed)
        m = make_model(X.shape[2], t, units, dropout, lr)
        h = m.fit(Xtr, ytr, sample_weight=sw, epochs=max_ep, batch_size=64, verbose=0,
                  validation_data=(Xva[known], yva[known], swva[known]))
        hist.append([h.history["loss"], h.history["val_loss"]])
    return np.array(hist).mean(0)

print("ready")

In [ ]:
# 4. Load the deployed v2 model and check it reproduces the deployed results
MODEL_DIR = "/content/drive/MyDrive/flood_tuning/v2_model"
if not os.path.exists(f"{MODEL_DIR}/config.json"):
    print("Upload config.json and the eight lstm_raw_seed*.weights.h5 files from models/flood_lstm/")
    os.makedirs("/content/v2_model", exist_ok=True)
    for name, data in files.upload().items():
        open(f"/content/v2_model/{name}", "wb").write(data)
    MODEL_DIR = "/content/v2_model"
cfg = json.load(open(f"{MODEL_DIR}/config.json"))
print("Model:", cfg.get("name", "lstm_raw"), cfg["version"], "| threshold", cfg["threshold"], "| window", cfg["window"])

X, meta, feats = make_windows(cfg["window"])
X = X[:, :, [feats.index(f) for f in cfg["features"]]]
test = meta["year"].isin(FINAL_FOLD[1]).values
Xte = ((X[test] - np.array(cfg["mean"])) / np.array(cfg["std"])).astype("float32")
score = np.zeros(len(Xte))
for seed in cfg["seeds"]:
    m = make_model(X.shape[2], cfg["window"], cfg.get("units", 32), cfg.get("dropout", 0.3), 0.001)
    m.load_weights(f"{MODEL_DIR}/lstm_raw_seed{seed}.weights.h5")
    score += m.predict(Xte, batch_size=1024, verbose=0).ravel() / len(cfg["seeds"])
s = meta.loc[test, ["catchment", "date"]].reset_index(drop=True)
s["val"] = "2024-2025"; s["score"] = score
s.to_csv(f"{OUT_DIR}/v2_test_scores.csv", index=False)
parts = prepare(s)
thr = cfg["threshold"]
r = {"ap": pooled_ap(parts), **stats(parts, thr, 6)}
r["tn"] = int(sum(len(p[2]) for p in parts)) - r["tp"] - r["fp"] - r["fn"]
r["precision_ci"] = wilson(r["tp"], r["tp"] + r["fp"]); r["recall_ci"] = wilson(r["tp"], r["tp"] + r["fn"])
r["event_recall_ci"] = wilson(r["detected"], r["events"])
json.dump(r, open(f"{OUT_DIR}/v2_results.json", "w"), indent=2, default=float)
print(pd.Series(r).to_string())
print(); print("Expected from the deployed evaluation: AP 0.533, TP 40, FP 26, FN 56, 20 of 38 events.")

In [ ]:
# 5. Confusion matrix at the deployed threshold (labelled days, 2024-2025)
import matplotlib.pyplot as plt
cm = np.array([[r["tp"], r["fn"]], [r["fp"], r["tn"]]])
pd.DataFrame(cm, index=["Actual flood", "Actual no flood"], columns=["Predicted flood", "Predicted no flood"]).to_csv(
    f"{OUT_DIR}/v2_confusion_matrix.csv")
fig, ax = plt.subplots(figsize=(5.2, 4.2), dpi=150)
shade = np.array([[1, 0], [0, 1]])
ax.imshow(shade, cmap=plt.matplotlib.colors.ListedColormap(["#fbe3d8", "#d6e6f8"]))
labels = [["True positive", "False negative"], ["False positive", "True negative"]]
for i in range(2):
    for j in range(2):
        ax.text(j, i - 0.08, f"{cm[i, j]:,}", ha="center", va="center", fontsize=18, fontweight="bold", color="#0b0b0b")
        ax.text(j, i + 0.22, labels[i][j], ha="center", va="center", fontsize=9, color="#52514e")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Predicted flood", "Predicted no flood"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Actual flood", "Actual no flood"])
ax.tick_params(length=0)
for sp in ax.spines.values(): sp.set_visible(False)
ax.set_title(f"lstm_raw v2 at threshold {thr}, 2024-2025", fontsize=10)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/v2_confusion_matrix.png"); plt.show()

In [ ]:
# 6. Training and validation loss of the v2 setting over its 15 epochs (4 seeds, averaged)
#    Left: development fold, trained on 2015-2022 and validated on 2023.
#    Right: the deployed training years 2015-2023, validated on 2024-2025. The model was already fixed, so this is
#    for reporting only and was not used to choose anything.
EP = 15
V2 = dict(t=cfg["window"], units=cfg.get("units", 32), dropout=cfg.get("dropout", 0.3), lr=0.001)
dev = loss_history(V2["t"], V2["units"], V2["dropout"], V2["lr"], EP, DEV_FOLDS[2], seeds=range(4))
fin = loss_history(V2["t"], V2["units"], V2["dropout"], V2["lr"], EP, FINAL_FOLD, seeds=range(4))
pd.DataFrame({"epoch": range(1, EP + 1), "dev_train_loss": dev[0], "dev_val_loss": dev[1],
              "final_train_loss": fin[0], "test_val_loss": fin[1]}).to_csv(f"{OUT_DIR}/v2_loss_curves.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), dpi=150)
for ax, h, title, vlabel in [(axes[0], dev, "Development fold", "Validation loss (2023)"),
                            (axes[1], fin, "Deployed training years", "Validation loss (2024-2025)")]:
    ax.plot(range(1, EP + 1), h[0], color="#2a78d6", lw=2, label="Training loss")
    ax.plot(range(1, EP + 1), h[1], color="#eb6834", lw=2, label=vlabel)
    ax.set_title(title, fontsize=10); ax.set_xlabel("Epoch"); ax.legend(frameon=False, fontsize=9)
    for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
axes[0].set_ylabel("Weighted cross-entropy")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/v2_loss_curves.png"); plt.show()
print(pd.read_csv(f"{OUT_DIR}/v2_loss_curves.csv").round(3).to_string(index=False))
with zipfile.ZipFile("/content/v2_report.zip", "w") as z:
    for f in os.listdir(OUT_DIR):
        z.write(f"{OUT_DIR}/{f}", f)
files.download("/content/v2_report.zip")